# Data catalog and quality

Your check of step 1.8. First run `fefat data quality` in a terminal: it writes the quality report of every product in your store, a few seconds each (`fefat data quality MES` writes MES's alone). Then pick the `.venv` kernel and run all cells. The first cell lists what your store holds, with each product's quality; the second shows MES's report, whose coverage table should match what you imported (`docs/data-inventory.md`); the third loads a quarter of MES 5-minute bars and times the call.

In [ ]:
import os
import time

import polars as pl
from IPython.display import Markdown, display

import fefat as ff
from fefat.config import resolve_paths

ROOT = "MES"

listed = ff.market.catalog()
listed

In [ ]:
provider = listed.filter(pl.col("root") == ROOT).item(0, "provider")
report = resolve_paths(os.environ).reports_dir / provider / ROOT / "quality.md"
display(Markdown(report.read_text("utf-8")))

In [ ]:
started = time.perf_counter()
bars = ff.market.load(ROOT, "5m", "2024-01-01", "2024-03-31").collect()
print(f"{bars.height:,} bars in {time.perf_counter() - started:.2f} s; run the cell again for the warm time.")
bars.head()

## What to look for

- The catalog lists every product you imported: MES from 2019-05-06 to 2026-09-23 at 1-minute fidelity, with its quality.
- In the report's coverage table, each month has as many 1m days as sessions; a month with fewer has gaps, which the findings list with their trade dates.
- The load takes under 2 seconds warm, and the bars have the columns of `ff.market.SIGNAL_BAR_COLUMNS`, with `roll_flag` on the first bar of MESM2024 on 2024-03-12.